# Checkpoint selection for the fine-tuned Qwen3-4B (Milestone 3, validation only)

Evaluates the four candidate checkpoints of `docs/milestone-2/EVAL_PROTOCOL.md`, Amendment 1, on the 300-case selection subset (`docs/milestone-3/selection-subset-v1`: 200 tier 1 and 100 tier 2 validation primaries), and chooses one: the highest S = ½(A₁ + A₂) − W, with exact ties going to the earlier checkpoint. **Validation data only; no test split is touched.**

For each candidate, `python -m packages.support_evaluator.checkpoint_selection evaluate`:

1. checks `state.json` against the sha256 logged when the checkpoint was saved, and the adapter files against `state.json`;
2. merges the LoRA adapter into the fp16 base (`Qwen/Qwen3-4B-Instruct-2507` at the pinned revision);
3. evaluates with the declared settings: the model's chat template, greedy decoding, 128 new tokens, prompt version 2, one generation at a time, on **one visible T4**;
4. runs the fp16 smoke test first (10 seeded subset cases per tier). The subset run starts only if it passes;
5. scores the 300 subset cases and writes `selection.json`: S, A₁, A₂, W, the monitored rates, the platform, GPU, library versions, bundle commit, adapter sha256 and every output file's sha256.

All four candidates must run on the **same platform with the same library versions**. `choose` refuses to compare them otherwise.

**On Kaggle (T4 x2):** the two GPUs evaluate two candidates at a time. Each evaluation process sees exactly one T4 (`CUDA_VISIBLE_DEVICES=0` or `1`), as the protocol requires; nothing is shared between them. **On Colab:** one T4, one candidate after another, with outputs on Google Drive so a disconnected session resumes where it stopped.

**Setup, Kaggle:**

- Accelerator *GPU T4 x2*, Internet *on*.
- Inputs: the evaluation bundle dataset, and the two training notebooks' outputs: session 1, which holds `checkpoint-0264` and `checkpoint-0528`, and the session 2 retry, which holds `checkpoint-0792` and `checkpoint-1055`.
- Set `BUNDLE_COMMIT`, then *Save Version → Save & Run All*. In the notebook's settings, keep the environment pinned to the original one (*Environment → Pin to original environment*) so that a later session uses the same library versions.

**Setup, Colab:**

- *Runtime → Change runtime type → T4 GPU*.
- On Google Drive, under `MyDrive/hetren-m3/`, put the bundle and a folder `checkpoints/` with `checkpoint-0264/`, `checkpoint-0528/`, `checkpoint-0792/` and `checkpoint-1055/`. Each needs only its `state.json` and `adapter/` folder from the training archives.
- Set `PLATFORM = "colab"` and `BUNDLE_COMMIT`, then *Run all*. After a disconnect, run all again: it resumes.

**If one session cannot run all four** (quota or time): set `RUN` to the candidates for this session. On Kaggle, attach this notebook's earlier output in the next session; finished and partial candidates are carried over and resumed. `choose` runs once all four are complete.

In [ ]:
PLATFORM = "kaggle"  # "kaggle" or "colab"
BUNDLE_COMMIT = ""  # the evaluation bundle's commit (full sha); the notebook stops if the bundle found is another
RUN = [264, 528, 792, 1055]  # the candidates to evaluate in this session
# state.json sha256 of each candidate, as logged in docs/milestone-3/VALIDATION_LOG.md when it was saved
STATE_SHA256 = {
    264: "ae7e31e7c23181e4cf30ff55da71210212a0dc63477b7627ce2f9efd24361273",
    528: "613ac6d5e4595a56752804a93701dadb538b06d9dd30b462feaccfb377988315",
    792: "d6230ff6e1780ac5286f0accc0a45298dd8db49fc11ed6fdb1330d98189276d5",
    1055: "54e0cc8820802e2b2a804616a62cc28c406c1c9e2dbe8e32bf3d4063e08b4b91",
}
DRIVE = "/content/drive/MyDrive/hetren-m3"  # Colab only

import glob, hashlib, json, os, shutil, subprocess, sys, tarfile, threading, time

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    INPUTS, WORK, OUT = DRIVE, "/content/hetren", f"{DRIVE}/qlora-selection"
else:
    INPUTS, WORK, OUT = "/kaggle/input", "/kaggle/working/hetren", "/kaggle/working/qlora-selection"


def earlier_session(path):  # a copy of the code inside an attached earlier notebook output, not the bundle itself
    return bool(glob.glob(f"{path.split('/hetren/')[0]}/qlora-*")) if "/hetren/" in path else False


if not os.path.exists(WORK):
    unpacked = [p for p in glob.glob(f"{INPUTS}/**/BUNDLE.json", recursive=True) if not earlier_session(p)]
    packed = [p for p in glob.glob(f"{INPUTS}/**/hetren-*bundle.tar.gz", recursive=True) if not earlier_session(p)]

    if len(unpacked) > 1 or (not unpacked and len(packed) != 1):
        raise SystemExit(f"STOP: attach exactly one bundle; found {unpacked or packed}")

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

    print("bundle from", unpacked[0] if unpacked else packed[0])

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)

if BUNDLE_COMMIT and COMMIT != BUNDLE_COMMIT.strip():
    raise SystemExit(f"STOP: the bundle is commit {COMMIT}, not BUNDLE_COMMIT {BUNDLE_COMMIT!r}")

os.chdir(WORK)

In [ ]:
# The evaluation libraries: transformers 4.56.2 (Amendment 1) and the versions used in training; torch is the platform's.
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
GPUS = [line.split(",")[1].strip() for line in subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"], capture_output=True, text=True).stdout.splitlines()]

if not GPUS or any("T4" not in name for name in GPUS):
    raise SystemExit(f"STOP: the GPUs are {GPUS}; the protocol evaluates on T4s")

print(len(GPUS), "T4s; each evaluation process sees one")

## Checkpoints, checked by hash, and the base model downloaded once

In [ ]:
CHECKPOINTS = {}

for step in RUN:
    found = sorted(glob.glob(f"{INPUTS}/**/checkpoint-{step:04d}/state.json", recursive=True))
    matching = [p for p in found if hashlib.sha256(open(p, "rb").read()).hexdigest() == STATE_SHA256[step]]

    if not matching:
        raise SystemExit(f"STOP: no checkpoint-{step:04d} with state.json sha256 {STATE_SHA256[step]} among {found}")

    CHECKPOINTS[step] = os.path.dirname(matching[0])
    print(f"checkpoint-{step:04d}: {CHECKPOINTS[step]}")

# Results of earlier sessions (attached outputs of this notebook) are carried over, complete or partial, to resume.
for earlier in glob.glob(f"{INPUTS}/**/qlora-selection/checkpoint-*", recursive=True):
    target = f"{OUT}/{os.path.basename(earlier)}"

    if not os.path.exists(target) and PLATFORM == "kaggle":
        shutil.copytree(earlier, target)
        print("carried over", earlier)

subprocess.run([sys.executable, "-c", "from huggingface_hub import snapshot_download; "
                "from packages.hetren_baselines.local_model import QWEN_MODEL, QWEN_REVISION; "
                "print(snapshot_download(QWEN_MODEL, revision=QWEN_REVISION))"], check=True)

## Evaluation (about 1.2 hours per candidate; two at a time on Kaggle's two T4s)

Each worker evaluates its candidates one after another on its own GPU. Logs: `qlora-selection/logs/checkpoint-<step>.log`.

In [ ]:
os.makedirs(f"{OUT}/logs", exist_ok=True)
pending = [s for s in RUN if not os.path.exists(f"{OUT}/checkpoint-{s:04d}/selection.json")]
EXITS = {}
SCHEDULE = f"{OUT}/schedule.jsonl"  # which candidate ran on which GPU and when (for the validation log)
utc = lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
lock = threading.Lock()


def worker(gpu, steps):
    for step in steps:
        name = f"checkpoint-{step:04d}"
        command = [sys.executable, "-m", "packages.support_evaluator.checkpoint_selection", "evaluate",
                   "--checkpoint", CHECKPOINTS[step], "--state-sha256", STATE_SHA256[step], "--out", f"{OUT}/{name}"]

        with open(f"{OUT}/logs/{name}.log", "a") as log:
            started, began = time.time(), utc()
            EXITS[step] = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                         env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}).returncode

        with lock, open(SCHEDULE, "a") as schedule:
            schedule.write(json.dumps({"checkpoint": name, "gpu": gpu, "gpu_name": GPUS[gpu], "started": began,
                                       "finished": utc(), "exit": EXITS[step], "platform": PLATFORM}) + "\n")

        print(f"{name} on GPU {gpu}: exit {EXITS[step]} after {(time.time() - started) / 3600:.2f} h", flush=True)


threads = [threading.Thread(target=worker, args=(gpu, pending[gpu::len(GPUS)])) for gpu in range(len(GPUS))]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

for step in RUN:
    path = f"{OUT}/checkpoint-{step:04d}/selection.json"

    if os.path.exists(path):
        result = json.load(open(path))
        print(f"checkpoint-{step:04d}: S {result['S']}  A1 {result['A1']['k']}/{result['A1']['n']}  "
              f"A2 {result['A2']['k']}/{result['A2']['n']}  W {result['W']['k']}/{result['W']['n']}  "
              f"calls {result['calls']['total']} (non-finite {result['calls']['non_finite']})")
    else:
        print(f"checkpoint-{step:04d}: not complete (exit {EXITS.get(step)}); see logs/checkpoint-{step:04d}.log")

## Choice (once all four candidates are complete) and outputs

In [ ]:
complete = sorted(glob.glob(f"{OUT}/checkpoint-*/selection.json"))

if len(complete) == 4:
    subprocess.run([sys.executable, "-m", "packages.support_evaluator.checkpoint_selection", "choose", "--out", OUT],
                   check=True)
else:
    print(f"{len(complete)} of 4 candidates complete; choose runs when all four are")

archive = ("/kaggle/working" if PLATFORM == "kaggle" else DRIVE) + "/qlora-selection.tar.gz"

with tarfile.open(archive, "w:gz") as results:
    results.add(OUT, arcname="qlora-selection")

print("archive", archive, hashlib.sha256(open(archive, "rb").read()).hexdigest())

if any(code != 0 for code in EXITS.values()):
    raise SystemExit(f"an evaluation stopped: {EXITS}; log it before re-running")